# Models comparisons
This is a notebook where the raw results of each model in each dataset is calculated and stored for later comparisons. We will compare our three best models (estimated in experiment1) with its local version, its centralized version and 4 more state of the art algorithms (FRF, DistBoost.F,PreWeak.F and AdaBoost.F). 10 datasets will be used. 

In [ ]:
from flextrees.datasets.tabular_datasets import bank
from flextrees.datasets.tabular_datasets import nursery
from flextrees.datasets.tabular_datasets import adult
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.datasets import load_breast_cancer
import time
import os
from models.FL_AdaBoost_Dist import FLEnsembleDist
from models.AdaBoostClassifier2 import AdaBoostClassifier2
from sklearn.metrics import accuracy_score,f1_score
from models.FRF import FRF_eval
from models.PolatoAdaBoost import polato_AdaBoost_eval

In [ ]:
train_data, test_data = bank(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data_bank = np.concatenate((X_data,X_test)).astype(int) #Some were bools
targets_bank = np.concatenate((y_data,y_test))
train_data, test_data = adult(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data_adult = np.concatenate((X_data,X_test))
targets_adult = np.concatenate((y_data,y_test))
train_data, test_data = nursery(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data_nursery = np.concatenate((X_data,X_test))
targets_nursery = np.concatenate((y_data,y_test))
filter = targets_nursery == 2
new_targets_nursery = targets_nursery[~filter]
new_X_data_nursery = X_data_nursery[~filter,:]
targets_nursery[targets_nursery==4] = 2

In [ ]:
np.unique(targets_nursery,return_counts=True)

In [ ]:
dataset = pd.read_csv('ildp.csv')
y_data_ildp = dataset.iloc[:,-1].to_numpy()
X_data_ildp = dataset.iloc[:,:-1].to_numpy()

In [ ]:
#ilpd_indian_liver_patient_dataset = fetch_ucirepo(id=225)
#X_data = ilpd_indian_liver_patient_dataset.data.features
## Preprocess the dataset. Transform the gender to a binary feature.
#X_data['Gender'] = X_data.apply(lambda row: 1 if 'Female' in row['Gender'] else 0, axis=1)
## Drop the rows with NaN values
#X_data = X_data.dropna(axis=0)
## Get the index of the rows that are not NaN
#X_data_index = X_data.index.to_list()
#X_data_ildp = X_data.to_numpy()
## Get the target values
#y_data = ilpd_indian_liver_patient_dataset.data.targets.iloc[X_data_index].to_numpy()
#y_data_ildp = np.array([y_real-1 for y_real in y_data]).flatten()

In [ ]:
np.unique(y_data_ildp,return_counts=True)

In [ ]:
dataset = pd.read_csv('creditcard.csv')
y_data_credit = dataset.iloc[:,-1].to_numpy()
X_data_credit = dataset.iloc[:,:-1].to_numpy()

In [ ]:
np.unique(y_data_credit,return_counts=True)

In [ ]:
dataset = pd.read_csv('magic.csv')
targets_magic = dataset.iloc[:,-1].to_numpy()
X_data_magic = dataset.iloc[:,:-1].to_numpy()

train_data, test_data = magic(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data_magic = np.concatenate((X_data,X_test))
targets_magic = np.concatenate((y_data,y_test))

In [ ]:
np.unique(targets_magic,return_counts=True)

In [ ]:
def preprocess_car_no_dict(row):
        col_changes = {
            'buying': {
                'vhigh': 0,
                'high': 1,
                'med': 2,
                'low': 3,
            },
            'maint': {
                'vhigh': 0,
                'high': 1,
                'med': 2,
                'low': 3,
            },
            'doors': {
                '2': 2,
                '3': 3,
                '4': 4,
                '5more': 5
            },
            'persons': {
                '2': 2,
                '4': 4,
                'more': 5,
            },
            'lug_boot': {
                'small': 0,
                'med': 1,
                'big': 2,
            },
            'safety': {
                'low': 0,
                'med': 1,
                'high': 2,
            },
        }
        return pd.Series([col_changes[feature][row[feature]] for feature in list(row.index[:-1])])

In [ ]:
dataset = pd.read_csv('car.csv',sep=',',names=['buying','maint','doors','persons','lug_boot','safety','target'])
col_names = list(dataset.columns)
c = {'unacc': 0, 'acc': 1, 'good': 2, 'vgood': 3}
y_data_car = dataset.get('target').apply(lambda x: c[x]).to_numpy()
X_data = dataset.apply(lambda x: preprocess_car_no_dict(x), axis=1)
X_data_car = X_data.to_numpy()

In [ ]:
np.unique(y_data_car,return_counts=True)

In [ ]:
dataset = pd.read_csv('covtype.csv',header =None)
y_data_covertype = dataset.iloc[:,54].to_numpy()
X_data_covertype = dataset.iloc[:,:54].to_numpy()
y_data_covertype = y_data_covertype-1

In [ ]:
np.unique(y_data_covertype,return_counts=True)

In [ ]:
breast_cancer = load_breast_cancer()
X_data_bcancer = breast_cancer.data
targets_bcancer = breast_cancer.target

In [ ]:
dataset_train = pd.read_csv('pendigitstra.csv',header=None)
dataset_test = pd.read_csv('pendigitstes.csv',header=None)
dataset = pd.concat([dataset_test,dataset_train],axis=0)
y_data_pendig = dataset.iloc[:,16].to_numpy()
X_data_pendig = dataset.iloc[:,:16].to_numpy()

In [ ]:
np.unique(y_data_pendig,return_counts=True)

In [ ]:
def compare_models(datasets,data_distrib,data_param,Nclients,seeds):
    acc_results1 = []
    f1_results1 = []
    acc_results2 = []
    f1_results2 = []
    acc_results3 = []
    f1_results3 = []
    times1 = []
    times2 = []
    times3 = []
    for dataset,data in datasets.items():
        fulldata,fulltargets = data
        for seed in seeds:
            X_data,public_data,y_data,public_data_targets = train_test_split(fulldata,fulltargets,
                                                                            train_size=0.75,random_state=seed)
            #Create folds
            skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=seed)
            indices_iterable = skf.split(X_data,y_data)
            
            dataset_splits = [None]*5
            for i,(filter_train,filter_test) in enumerate(indices_iterable):
                dataset_splits[i] = (X_data[filter_train],X_data[filter_test],
                                    y_data[filter_train],y_data[filter_test])
            acc_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['o_s_Wmv_cw_global','o_s_Wmv_cw_local','o_s_Wmv_aw_global',
                                                                                                             'o_s_Wmv_aw_local','s_c_b_nb_Wmv_aw_global','s_c_b_nb_Wmv_aw_local'])
            f1_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['o_s_Wmv_cw_global','o_s_Wmv_cw_local','o_s_Wmv_aw_global',
                                                                                                             'o_s_Wmv_aw_local','s_c_b_nb_Wmv_aw_global','s_c_b_nb_Wmv_aw_local'])
            acc_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['centralized_global','centralized_local','loc_model_global','loc_model_local'])
            f1_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['centralized_global','centralized_local','loc_model_global','loc_model_local'])
            acc_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['FRF_global','FRF_local','DistBoost_global','DistBoost_local','PreWeak_global',
                                                                                                             'PreWeak_local','AdaBoost_global','AdaBoost_local'])
            f1_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns =  ['FRF_global','FRF_local','DistBoost_global','DistBoost_local','PreWeak_global',
                                                                                                             'PreWeak_local','AdaBoost_global','AdaBoost_local'])
            
            times_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['o_s_Wmv_cw','o_s_Wmv_aw','s_c_b_nb_Wmv_aw'])
            times_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['centralized','loc_model'])
            times_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)]*len(dataset_splits),columns = ['FRF','DistBoost','PreWeak','AdaBoost'])
            
            # train each algorithm with every split

            for i,data in enumerate(dataset_splits):
                X_train,X_test,y_train,y_test = data
                o_s_Wmv_cw = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                            data_distribution=data_distrib,distribution_param=data_param,
                                            public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='common_weighted',
                                            prediction_weights='only_server',random_state=seed)
                #store data distrib for later models
                train_dict = o_s_Wmv_cw.train_clients_data.copy()
                test_dict = o_s_Wmv_cw.test_clients_data.copy()
                #Takes the columns of weights out
                for key,(train,labeltr) in train_dict.items():
                    train_dict[key] = (train[:,:-1],labeltr)
                    test,labelte = test_dict[key]
                    test_dict[key] = (test[:,:-1],labelte)

                time0 = time.time()
                o_s_Wmv_cw.fitmodel()
                os_cw_acc_global,os_cw_f1_global,os_cw_acc_local,os_cw_f1_local = o_s_Wmv_cw.sumar_overall_score(X_test,y_test)
                o_s_cw_time = time.time() - time0

                o_s_Wmv_aw = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                            data_distribution=data_distrib,distribution_param=data_param,
                                            public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='avg_weighted',
                                            prediction_weights='only_server',random_state=seed)
                time0 = time.time()
                o_s_Wmv_aw.fitmodel()
                os_aw_acc_global,os_aw_f1_global,os_aw_acc_local,os_aw_f1_local = o_s_Wmv_aw.sumar_overall_score(X_test,y_test)
                o_s_aw_time = time.time() - time0
                s_cl = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                            data_distribution=data_distrib,distribution_param=data_param,
                                            public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='common_weighted',
                                            prediction_weights='server_and_clients',random_state=seed)
                time0 = time.time()
                s_cl.fitmodel()
                s_cl_acc_global,s_cl_f1_global,s_cl_acc_local,s_cl_f1_local = s_cl.sumar_overall_score(X_test,y_test)
                s_cl_time = time.time() - time0

                #Stores results in dataframes
                acc_dataframe1.iloc[i,:] = np.array([os_cw_acc_global,os_cw_acc_local,os_aw_acc_global,os_aw_acc_local,s_cl_acc_global,s_cl_acc_local])
                f1_dataframe1.iloc[i,:] = np.array([os_cw_f1_global,os_cw_f1_local,os_aw_f1_global,os_aw_f1_local,s_cl_f1_global,s_cl_f1_local])
                times_dataframe1.iloc[i,:] = np.array([o_s_cw_time,o_s_aw_time,s_cl_time])

                #Train centralized and local models and get scores:
                time0 = time.time()
                centralized_model = AdaBoostClassifier2(n_estimators=10,random_state=seed+3*Nclients,max_depth=8)
                centralized_model.fit(X_train,y_train)
                y_pred = centralized_model.predict(X_test)
                cent_acc_global = accuracy_score(y_pred,y_test)*100
                cent_f1_global = f1_score(y_pred,y_test,labels=np.unique(y_test),average='weighted',zero_division=0.0)*100
                acc_scores = np.zeros(Nclients)
                f1_scores = np.zeros(Nclients)
                for key,(test_data,ytest_data) in test_dict.items():
                    y_pred = centralized_model.predict(test_data)
                    acc_scores[key] = accuracy_score(y_pred,ytest_data)*100
                    f1_scores[key] = f1_score(y_pred,ytest_data,labels=np.unique(ytest_data),average='weighted',zero_division=0.0)*100
                cent_acc_local = acc_scores.mean()
                cent_f1_local = f1_scores.mean()
                cent_time = time.time()-time0

                time0 = time.time()
                acc_scores_local = np.zeros(Nclients)
                acc_scores_global = np.zeros(Nclients)
                f1_scores_local = np.zeros(Nclients)
                f1_scores_global = np.zeros(Nclients)
                for key,(train,labeltr) in test_dict.items():
                    test,labelte = test_dict[key]
                    model = AdaBoostClassifier2(n_estimators=10,random_state=seed+key,max_depth=8)
                    model.fit(train,labeltr)
                    y_pred_global = model.predict(X_test)
                    y_pred_local = model.predict(test)
                    acc_scores_global[key] = accuracy_score(y_pred_global,y_test)*100
                    acc_scores_local[key] = accuracy_score(y_pred_local,labelte)*100
                    f1_scores_global[key] = f1_score(y_pred_global,y_test,labels = np.unique(y_test),average='weighted',zero_division=0.0)*100
                    f1_scores_local[key] = f1_score(y_pred_local,labelte,labels = np.unique(labelte),average='weighted',zero_division=0.0)*100
                local_acc_local = acc_scores_local.mean()
                local_acc_global = acc_scores_global.mean()
                local_f1_local = f1_scores_local.mean()
                local_f1_global = f1_scores_global.mean()
                local_time = time.time()-time0

                acc_dataframe2.iloc[i,:] = np.array([cent_acc_global,cent_acc_local,local_acc_global,local_acc_local])
                f1_dataframe2.iloc[i,:] = np.array([cent_f1_global,cent_f1_local,local_f1_global,local_f1_local])

                times_dataframe2.iloc[i,:] = np.array([cent_time,local_time])

                #Other models: 
                time0 = time.time()
                FRF_acc_global,FRF_f1_global,FRF_acc_local,FRF_f1_local = FRF_eval(train_dict,test_dict,X_test,y_test,hyperparameters='theirs')
                FRF_time = time.time()-time0
                time0 = time.time()
                DB_acc_global,DB_f1_global,DB_acc_local,DB_f1_local = polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='distsamme',n_estimators=300,max_leaf_nodes=10)
                DB_time = time.time()-time0
                time0 = time.time()
                PW_acc_global,PW_f1_global,PW_acc_local,PW_f1_local= polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='preweaksamme',n_estimators=300,max_leaf_nodes=10)
                PW_time = time.time()-time0
                time0 = time.time()
                AB_acc_global,AB_f1_global,AB_acc_local,AB_f1_local = polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='adaboost.f1',n_estimators=300,max_leaf_nodes=10)
                AB_time = time.time()-time0

                acc_dataframe3.iloc[i,:] = np.array([FRF_acc_global,FRF_acc_local,DB_acc_global,DB_acc_local,PW_acc_global,PW_acc_local,AB_acc_global,AB_acc_local])
                f1_dataframe3.iloc[i,:] = np.array([FRF_f1_global,FRF_f1_local,DB_f1_global,DB_f1_local,PW_f1_global,PW_f1_local,AB_f1_global,AB_f1_local])
                times_dataframe3.iloc[i,:] = np.array([FRF_time,DB_time,PW_time,AB_time])
            acc_results1.append(acc_dataframe1)
            f1_results1.append(f1_dataframe1)
            times1.append(times_dataframe1)
            acc_results2.append(acc_dataframe2)
            f1_results2.append(f1_dataframe2)
            times2.append(times_dataframe2)
            acc_results3.append(acc_dataframe3)
            f1_results3.append(f1_dataframe3)
            times3.append(times_dataframe3)
    
    my_models_accdataframe = pd.concat(acc_results1,axis=0)
    my_models_accdataframe = my_models_accdataframe.rename_axis(data_distrib + '_Mymodels_acc_'+ str(Nclients))
    my_models_f1dataframe = pd.concat(f1_results1,axis=0)
    my_models_f1dataframe = my_models_f1dataframe.rename_axis(data_distrib +'_Mymodels_f1_'+ str(Nclients))
    my_models_times = pd.concat(times1,axis=0)
    my_models_times = my_models_times.rename_axis(data_distrib +'_Mymodel_times_' + str(Nclients))
    notfed_accdataframe = pd.concat(acc_results2,axis=0)
    notfed_accdataframe = notfed_accdataframe.rename_axis(data_distrib +'_Notfed_acc_'+ str(Nclients))
    notfed_f1dataframe = pd.concat(f1_results2,axis=0)
    notfed_f1dataframe = notfed_f1dataframe.rename_axis(data_distrib +'_notfed_f1_'+ str(Nclients))
    notfed_times = pd.concat(times2,axis=0)
    notfed_times = notfed_times.rename_axis(data_distrib +'_other_models_times_' + str(Nclients))
    other_models_accdataframe = pd.concat(acc_results3,axis=0)
    other_models_accdataframe = other_models_accdataframe.rename_axis(data_distrib +'_other_models_acc_'+ str(Nclients))
    other_models_f1dataframe = pd.concat(f1_results3,axis=0)
    other_models_f1dataframe = other_models_f1dataframe.rename_axis(data_distrib +'_other_models_f1_'+ str(Nclients))
    other_models_times = pd.concat(times3,axis=0)
    other_models_times = other_models_times.rename_axis(data_distrib +'_other_models_times_' + str(Nclients))

    return my_models_accdataframe,my_models_f1dataframe,my_models_times,notfed_accdataframe,notfed_f1dataframe,notfed_times,other_models_accdataframe,other_models_f1dataframe,other_models_times

In [ ]:
def compare_models_woutXval(datasets,data_distrib,data_param,Nclients,seeds):
    acc_results1 = []
    f1_results1 = []
    acc_results2 = []
    f1_results2 = []
    acc_results3 = []
    f1_results3 = []
    times1 = []
    times2 = []
    times3 = []
    for dataset,data in datasets.items():
        fulldata,fulltargets = data
        for seed in seeds:
            X_data,public_data,y_data,public_data_targets = train_test_split(fulldata,fulltargets,
                                                                            train_size=0.75,random_state=seed)
            X_train,X_test,y_train,y_test = train_test_split(X_data,y_data,random_state = seed)
            acc_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['o_s_Wmv_cw_global','o_s_Wmv_cw_local','o_s_Wmv_aw_global',
                                                                                                             'o_s_Wmv_aw_local','s_c_b_nb_Wmv_aw_global','s_c_b_nb_Wmv_aw_local'])
            f1_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['o_s_Wmv_cw_global','o_s_Wmv_cw_local','o_s_Wmv_aw_global',
                                                                                                             'o_s_Wmv_aw_local','s_c_b_nb_Wmv_aw_global','s_c_b_nb_Wmv_aw_local'])
            acc_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['centralized_global','centralized_local','loc_model_global','loc_model_local'])
            f1_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['centralized_global','centralized_local','loc_model_global','loc_model_local'])
            acc_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['FRF_global','FRF_local','DistBoost_global','DistBoost_local','PreWeak_global',
                                                                                                             'PreWeak_local','AdaBoost_global','AdaBoost_local'])
            f1_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns =  ['FRF_global','FRF_local','DistBoost_global','DistBoost_local','PreWeak_global',
                                                                                                             'PreWeak_local','AdaBoost_global','AdaBoost_local'])
            
            times_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['o_s_Wmv_cw','o_s_Wmv_aw','s_c_b_nb_Wmv_aw'])
            times_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['centralized','loc_model'])
            times_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['FRF','DistBoost','PreWeak','AdaBoost'])
            
            
            o_s_Wmv_cw = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                        data_distribution=data_distrib,distribution_param=data_param,
                                        public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='common_weighted',
                                        prediction_weights='only_server',random_state=seed)
            #store data distrib for later models
            train_dict = o_s_Wmv_cw.train_clients_data.copy()
            test_dict = o_s_Wmv_cw.test_clients_data.copy()
            #Takes the columns of weights out
            for key,(train,labeltr) in train_dict.items():
                train_dict[key] = (train[:,:-1],labeltr)
                test,labelte = test_dict[key]
                test_dict[key] = (test[:,:-1],labelte)

            time0 = time.time()
            o_s_Wmv_cw.fitmodel()
            os_cw_acc_global,os_cw_f1_global,os_cw_acc_local,os_cw_f1_local = o_s_Wmv_cw.sumar_overall_score(X_test,y_test)
            o_s_cw_time = time.time() - time0

            o_s_Wmv_aw = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                        data_distribution=data_distrib,distribution_param=data_param,
                                        public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='avg_weighted',
                                        prediction_weights='only_server',random_state=seed)
            time0 = time.time()
            o_s_Wmv_aw.fitmodel()
            os_aw_acc_global,os_aw_f1_global,os_aw_acc_local,os_aw_f1_local = o_s_Wmv_aw.sumar_overall_score(X_test,y_test)
            o_s_aw_time = time.time() - time0
            s_cl = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                        data_distribution=data_distrib,distribution_param=data_param,
                                        public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='common_weighted',
                                        prediction_weights='server_and_clients',random_state=seed)
            time0 = time.time()
            s_cl.fitmodel()
            s_cl_acc_global,s_cl_f1_global,s_cl_acc_local,s_cl_f1_local = s_cl.sumar_overall_score(X_test,y_test)
            s_cl_time = time.time() - time0

            #Stores results in dataframes
            acc_dataframe1.iloc[0,:] = np.array([os_cw_acc_global,os_cw_acc_local,os_aw_acc_global,os_aw_acc_local,s_cl_acc_global,s_cl_acc_local])
            f1_dataframe1.iloc[0,:] = np.array([os_cw_f1_global,os_cw_f1_local,os_aw_f1_global,os_aw_f1_local,s_cl_f1_global,s_cl_f1_local])
            times_dataframe1.iloc[0,:] = np.array([o_s_cw_time,o_s_aw_time,s_cl_time])

            #Train centralized and local models and get scores:
            time0 = time.time()
            centralized_model = AdaBoostClassifier2(n_estimators=10,random_state=seed+3*Nclients,max_depth=8)
            centralized_model.fit(X_train,y_train)
            y_pred = centralized_model.predict(X_test)
            cent_acc_global = accuracy_score(y_pred,y_test)*100
            cent_f1_global = f1_score(y_pred,y_test,labels=np.unique(y_test),average='weighted',zero_division=0.0)*100
            acc_scores = np.zeros(Nclients)
            f1_scores = np.zeros(Nclients)
            for key,(test_data,ytest_data) in test_dict.items():
                y_pred = centralized_model.predict(test_data)
                acc_scores[key] = accuracy_score(y_pred,ytest_data)*100
                f1_scores[key] = f1_score(y_pred,ytest_data,labels=np.unique(ytest_data),average='weighted',zero_division=0.0)*100
            cent_acc_local = acc_scores.mean()
            cent_f1_local = f1_scores.mean()
            cent_time = time.time()-time0

            time0 = time.time()
            acc_scores_local = np.zeros(Nclients)
            acc_scores_global = np.zeros(Nclients)
            f1_scores_local = np.zeros(Nclients)
            f1_scores_global = np.zeros(Nclients)
            for key,(train,labeltr) in test_dict.items():
                test,labelte = test_dict[key]
                model = AdaBoostClassifier2(n_estimators=10,random_state=seed+key,max_depth=8)
                model.fit(train,labeltr)
                y_pred_global = model.predict(X_test)
                y_pred_local = model.predict(test)
                acc_scores_global[key] = accuracy_score(y_pred_global,y_test)*100
                acc_scores_local[key] = accuracy_score(y_pred_local,labelte)*100
                f1_scores_global[key] = f1_score(y_pred_global,y_test,labels = np.unique(y_test),average='weighted',zero_division=0.0)*100
                f1_scores_local[key] = f1_score(y_pred_local,labelte,labels = np.unique(labelte),average='weighted',zero_division=0.0)*100
            local_acc_local = acc_scores_local.mean()
            local_acc_global = acc_scores_global.mean()
            local_f1_local = f1_scores_local.mean()
            local_f1_global = f1_scores_global.mean()
            local_time = time.time()-time0

            acc_dataframe2.iloc[0,:] = np.array([cent_acc_global,cent_acc_local,local_acc_global,local_acc_local])
            f1_dataframe2.iloc[0,:] = np.array([cent_f1_global,cent_f1_local,local_f1_global,local_f1_local])

            times_dataframe2.iloc[0,:] = np.array([cent_time,local_time])

            #Other models: 
            time0 = time.time()
            FRF_acc_global,FRF_f1_global,FRF_acc_local,FRF_f1_local = FRF_eval(train_dict,test_dict,X_test,y_test,hyperparameters='theirs')
            FRF_time = time.time()-time0
            time0 = time.time()
            DB_acc_global,DB_f1_global,DB_acc_local,DB_f1_local = polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='distsamme',n_estimators=300,max_leaf_nodes=10)
            DB_time = time.time()-time0
            time0 = time.time()
            PW_acc_global,PW_f1_global,PW_acc_local,PW_f1_local= polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='preweaksamme',n_estimators=300,max_leaf_nodes=10)
            PW_time = time.time()-time0
            time0 = time.time()
            AB_acc_global,AB_f1_global,AB_acc_local,AB_f1_local = polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='adaboost.f1',n_estimators=300,max_leaf_nodes=10)
            AB_time = time.time()-time0

            acc_dataframe3.iloc[0,:] = np.array([FRF_acc_global,FRF_acc_local,DB_acc_global,DB_acc_local,PW_acc_global,PW_acc_local,AB_acc_global,AB_acc_local])
            f1_dataframe3.iloc[0,:] = np.array([FRF_f1_global,FRF_f1_local,DB_f1_global,DB_f1_local,PW_f1_global,PW_f1_local,AB_f1_global,AB_f1_local])
            times_dataframe3.iloc[0,:] = np.array([FRF_time,DB_time,PW_time,AB_time])
            acc_results1.append(acc_dataframe1)
            f1_results1.append(f1_dataframe1)
            times1.append(times_dataframe1)
            acc_results2.append(acc_dataframe2)
            f1_results2.append(f1_dataframe2)
            times2.append(times_dataframe2)
            acc_results3.append(acc_dataframe3)
            f1_results3.append(f1_dataframe3)
            times3.append(times_dataframe3)
    
    my_models_accdataframe = pd.concat(acc_results1,axis=0)
    my_models_accdataframe = my_models_accdataframe.rename_axis(data_distrib + '_Mymodels_acc_'+ str(Nclients))
    my_models_f1dataframe = pd.concat(f1_results1,axis=0)
    my_models_f1dataframe = my_models_f1dataframe.rename_axis(data_distrib +'_Mymodels_f1_'+ str(Nclients))
    my_models_times = pd.concat(times1,axis=0)
    my_models_times = my_models_times.rename_axis(data_distrib +'_Mymodel_times_' + str(Nclients))
    notfed_accdataframe = pd.concat(acc_results2,axis=0)
    notfed_accdataframe = notfed_accdataframe.rename_axis(data_distrib +'_Notfed_acc_'+ str(Nclients))
    notfed_f1dataframe = pd.concat(f1_results2,axis=0)
    notfed_f1dataframe = notfed_f1dataframe.rename_axis(data_distrib +'_notfed_f1_'+ str(Nclients))
    notfed_times = pd.concat(times2,axis=0)
    notfed_times = notfed_times.rename_axis(data_distrib +'_other_models_times_' + str(Nclients))
    other_models_accdataframe = pd.concat(acc_results3,axis=0)
    other_models_accdataframe = other_models_accdataframe.rename_axis(data_distrib +'_other_models_acc_'+ str(Nclients))
    other_models_f1dataframe = pd.concat(f1_results3,axis=0)
    other_models_f1dataframe = other_models_f1dataframe.rename_axis(data_distrib +'_other_models_f1_'+ str(Nclients))
    other_models_times = pd.concat(times3,axis=0)
    other_models_times = other_models_times.rename_axis(data_distrib +'_other_models_times_' + str(Nclients))

    return my_models_accdataframe,my_models_f1dataframe,my_models_times,notfed_accdataframe,notfed_f1dataframe,notfed_times,other_models_accdataframe,other_models_f1dataframe,other_models_times

In [ ]:
def compare_models_woutXval_myparams(datasets,data_distrib,data_param,Nclients,seeds):
    acc_results1 = []
    f1_results1 = []
    acc_results2 = []
    f1_results2 = []
    acc_results3 = []
    f1_results3 = []
    times1 = []
    times2 = []
    times3 = []
    for dataset,data in datasets.items():
        fulldata,fulltargets = data
        for seed in seeds:
            X_data,public_data,y_data,public_data_targets = train_test_split(fulldata,fulltargets,
                                                                            train_size=0.75,random_state=seed)
            X_train,X_test,y_train,y_test = train_test_split(X_data,y_data,random_state = seed)
            acc_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['o_s_Wmv_cw_global','o_s_Wmv_cw_local','o_s_Wmv_aw_global',
                                                                                                             'o_s_Wmv_aw_local','s_c_b_nb_Wmv_aw_global','s_c_b_nb_Wmv_aw_local'])
            f1_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['o_s_Wmv_cw_global','o_s_Wmv_cw_local','o_s_Wmv_aw_global',
                                                                                                             'o_s_Wmv_aw_local','s_c_b_nb_Wmv_aw_global','s_c_b_nb_Wmv_aw_local'])
            acc_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['centralized_global','centralized_local','loc_model_global','loc_model_local'])
            f1_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['centralized_global','centralized_local','loc_model_global','loc_model_local'])
            acc_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['FRF_global','FRF_local','DistBoost_global','DistBoost_local','PreWeak_global',
                                                                                                             'PreWeak_local','AdaBoost_global','AdaBoost_local'])
            f1_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns =  ['FRF_global','FRF_local','DistBoost_global','DistBoost_local','PreWeak_global',
                                                                                                             'PreWeak_local','AdaBoost_global','AdaBoost_local'])
            
            times_dataframe1 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['o_s_Wmv_cw','o_s_Wmv_aw','s_c_b_nb_Wmv_aw'])
            times_dataframe2 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['centralized','loc_model'])
            times_dataframe3 = pd.DataFrame(np.nan, index = [dataset+str(seed)],columns = ['FRF','DistBoost','PreWeak','AdaBoost'])
            
            
            o_s_Wmv_cw = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                        data_distribution=data_distrib,distribution_param=data_param,
                                        public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='common_weighted',
                                        prediction_weights='only_server',random_state=seed)
            #store data distrib for later models
            train_dict = o_s_Wmv_cw.train_clients_data.copy()
            test_dict = o_s_Wmv_cw.test_clients_data.copy()
            #Takes the columns of weights out
            for key,(train,labeltr) in train_dict.items():
                train_dict[key] = (train[:,:-1],labeltr)
                test,labelte = test_dict[key]
                test_dict[key] = (test[:,:-1],labelte)

            time0 = time.time()
            o_s_Wmv_cw.fitmodel()
            os_cw_acc_global,os_cw_f1_global,os_cw_acc_local,os_cw_f1_local = o_s_Wmv_cw.sumar_overall_score(X_test,y_test)
            o_s_cw_time = time.time() - time0

            o_s_Wmv_aw = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                        data_distribution=data_distrib,distribution_param=data_param,
                                        public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='avg_weighted',
                                        prediction_weights='only_server',random_state=seed)
            time0 = time.time()
            o_s_Wmv_aw.fitmodel()
            os_aw_acc_global,os_aw_f1_global,os_aw_acc_local,os_aw_f1_local = o_s_Wmv_aw.sumar_overall_score(X_test,y_test)
            o_s_aw_time = time.time() - time0
            s_cl = FLEnsembleDist(X_train,y_train,public_data,Nclients=Nclients,max_depth=8,T=10,
                                        data_distribution=data_distrib,distribution_param=data_param,
                                        public_data_prediction='weighted_majority_voting',server_alpha_weight_adj='common_weighted',
                                        prediction_weights='server_and_clients',random_state=seed)
            time0 = time.time()
            s_cl.fitmodel()
            s_cl_acc_global,s_cl_f1_global,s_cl_acc_local,s_cl_f1_local = s_cl.sumar_overall_score(X_test,y_test)
            s_cl_time = time.time() - time0

            #Stores results in dataframes
            acc_dataframe1.iloc[0,:] = np.array([os_cw_acc_global,os_cw_acc_local,os_aw_acc_global,os_aw_acc_local,s_cl_acc_global,s_cl_acc_local])
            f1_dataframe1.iloc[0,:] = np.array([os_cw_f1_global,os_cw_f1_local,os_aw_f1_global,os_aw_f1_local,s_cl_f1_global,s_cl_f1_local])
            times_dataframe1.iloc[0,:] = np.array([o_s_cw_time,o_s_aw_time,s_cl_time])

            #Train centralized and local models and get scores:
            time0 = time.time()
            centralized_model = AdaBoostClassifier2(n_estimators=10,random_state=seed+3*Nclients,max_depth=8)
            centralized_model.fit(X_train,y_train)
            y_pred = centralized_model.predict(X_test)
            cent_acc_global = accuracy_score(y_pred,y_test)*100
            cent_f1_global = f1_score(y_pred,y_test,labels=np.unique(y_test),average='weighted',zero_division=0.0)*100
            acc_scores = np.zeros(Nclients)
            f1_scores = np.zeros(Nclients)
            for key,(test_data,ytest_data) in test_dict.items():
                y_pred = centralized_model.predict(test_data)
                acc_scores[key] = accuracy_score(y_pred,ytest_data)*100
                f1_scores[key] = f1_score(y_pred,ytest_data,labels=np.unique(ytest_data),average='weighted',zero_division=0.0)*100
            cent_acc_local = acc_scores.mean()
            cent_f1_local = f1_scores.mean()
            cent_time = time.time()-time0

            time0 = time.time()
            acc_scores_local = np.zeros(Nclients)
            acc_scores_global = np.zeros(Nclients)
            f1_scores_local = np.zeros(Nclients)
            f1_scores_global = np.zeros(Nclients)
            for key,(train,labeltr) in test_dict.items():
                test,labelte = test_dict[key]
                model = AdaBoostClassifier2(n_estimators=10,random_state=seed+key,max_depth=8)
                model.fit(train,labeltr)
                y_pred_global = model.predict(X_test)
                y_pred_local = model.predict(test)
                acc_scores_global[key] = accuracy_score(y_pred_global,y_test)*100
                acc_scores_local[key] = accuracy_score(y_pred_local,labelte)*100
                f1_scores_global[key] = f1_score(y_pred_global,y_test,labels = np.unique(y_test),average='weighted',zero_division=0.0)*100
                f1_scores_local[key] = f1_score(y_pred_local,labelte,labels = np.unique(labelte),average='weighted',zero_division=0.0)*100
            local_acc_local = acc_scores_local.mean()
            local_acc_global = acc_scores_global.mean()
            local_f1_local = f1_scores_local.mean()
            local_f1_global = f1_scores_global.mean()
            local_time = time.time()-time0

            acc_dataframe2.iloc[0,:] = np.array([cent_acc_global,cent_acc_local,local_acc_global,local_acc_local])
            f1_dataframe2.iloc[0,:] = np.array([cent_f1_global,cent_f1_local,local_f1_global,local_f1_local])

            times_dataframe2.iloc[0,:] = np.array([cent_time,local_time])

            #Other models: 
            time0 = time.time()
            FRF_acc_global,FRF_f1_global,FRF_acc_local,FRF_f1_local = FRF_eval(train_dict,test_dict,X_test,y_test,hyperparameters='theirs')
            FRF_time = time.time()-time0
            time0 = time.time()
            DB_acc_global,DB_f1_global,DB_acc_local,DB_f1_local = polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='distsamme',n_estimators=10,max_depth=8)
            DB_time = time.time()-time0
            time0 = time.time()
            PW_acc_global,PW_f1_global,PW_acc_local,PW_f1_local= polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='preweaksamme',n_estimators=10,max_depth=8)
            PW_time = time.time()-time0
            time0 = time.time()
            AB_acc_global,AB_f1_global,AB_acc_local,AB_f1_local = polato_AdaBoost_eval(train_dict,test_dict, X_test,y_test, seed, model='adaboost.f1',n_estimators=10,max_depth=8)
            AB_time = time.time()-time0

            acc_dataframe3.iloc[0,:] = np.array([FRF_acc_global,FRF_acc_local,DB_acc_global,DB_acc_local,PW_acc_global,PW_acc_local,AB_acc_global,AB_acc_local])
            f1_dataframe3.iloc[0,:] = np.array([FRF_f1_global,FRF_f1_local,DB_f1_global,DB_f1_local,PW_f1_global,PW_f1_local,AB_f1_global,AB_f1_local])
            times_dataframe3.iloc[0,:] = np.array([FRF_time,DB_time,PW_time,AB_time])
            acc_results1.append(acc_dataframe1)
            f1_results1.append(f1_dataframe1)
            times1.append(times_dataframe1)
            acc_results2.append(acc_dataframe2)
            f1_results2.append(f1_dataframe2)
            times2.append(times_dataframe2)
            acc_results3.append(acc_dataframe3)
            f1_results3.append(f1_dataframe3)
            times3.append(times_dataframe3)
        
    my_models_accdataframe = pd.concat(acc_results1,axis=0)
    my_models_accdataframe = my_models_accdataframe.rename_axis(data_distrib + '_Mymodels_acc_'+ str(Nclients))
    my_models_f1dataframe = pd.concat(f1_results1,axis=0)
    my_models_f1dataframe = my_models_f1dataframe.rename_axis(data_distrib +'_Mymodels_f1_'+ str(Nclients))
    my_models_times = pd.concat(times1,axis=0)
    my_models_times = my_models_times.rename_axis(data_distrib +'_Mymodel_times_' + str(Nclients))
    notfed_accdataframe = pd.concat(acc_results2,axis=0)
    notfed_accdataframe = notfed_accdataframe.rename_axis(data_distrib +'_Notfed_acc_'+ str(Nclients))
    notfed_f1dataframe = pd.concat(f1_results2,axis=0)
    notfed_f1dataframe = notfed_f1dataframe.rename_axis(data_distrib +'_notfed_f1_'+ str(Nclients))
    notfed_times = pd.concat(times2,axis=0)
    notfed_times = notfed_times.rename_axis(data_distrib +'_other_models_times_' + str(Nclients))
    other_models_accdataframe = pd.concat(acc_results3,axis=0)
    other_models_accdataframe = other_models_accdataframe.rename_axis(data_distrib +'_other_models_acc_'+ str(Nclients))
    other_models_f1dataframe = pd.concat(f1_results3,axis=0)
    other_models_f1dataframe = other_models_f1dataframe.rename_axis(data_distrib +'_other_models_f1_'+ str(Nclients))
    other_models_times = pd.concat(times3,axis=0)
    other_models_times = other_models_times.rename_axis(data_distrib +'_other_models_times_' + str(Nclients))

    return my_models_accdataframe,my_models_f1dataframe,my_models_times,notfed_accdataframe,notfed_f1dataframe,notfed_times,other_models_accdataframe,other_models_f1dataframe,other_models_times

In [ ]:
datasets = {'bank':(X_data_bank,targets_bank),
            'adult':(X_data_adult,targets_adult),'nursery':(X_data_nursery,targets_nursery),'bcancer':(X_data_bcancer,targets_bcancer),
            'ildp':(X_data_ildp,y_data_ildp),'credit':(X_data_credit,y_data_credit),'car':(X_data_car,y_data_car),
            'covertype':(X_data_covertype,y_data_covertype),'magic':(X_data_magic,targets_magic),'pendig':(X_data_pendig,y_data_pendig)}

In [ ]:
X_data_adult.shape

In [ ]:
abspath = os.path.abspath('')
numberOfClients = [5,10]
data_distribution = [('iid',None),('niid_quantity_skew',0.5),('niid_dirichlet_label_skew',0.5)]
for Nclients in numberOfClients:
    for data_distrib,data_param in data_distribution:
        results = compare_models_woutXval_myparams(datasets=datasets,data_distrib=data_distrib,data_param=data_param,
                                 Nclients=Nclients,seeds=[0,142,242,342,442,542])
        for result in results:
            path = abspath + '\\results' + str(Nclients) + 'clients\\'+ result.index.name
            result.to_csv(path_or_buf=path)

In [ ]:
#remove those datasets with not enough instances to split between 20 clients
datasets = {'bank':(X_data_bank,targets_bank),
            'adult':(X_data_adult,targets_adult),'nursery':(X_data_nursery,targets_nursery),
            'credit':(X_data_credit,y_data_credit),'car':(X_data_car,y_data_car),
            'covertype':(X_data_covertype,y_data_covertype),'magic':(X_data_magic,targets_magic),'pendig':(X_data_pendig,y_data_pendig)}

In [ ]:
abspath = os.path.abspath('')
numberOfClients = [20]
data_distribution = [('iid',None),('niid_quantity_skew',0.5),('niid_dirichlet_label_skew',0.5)]
for Nclients in numberOfClients:
    for data_distrib,data_param in data_distribution:
        results = compare_models_woutXval_myparams(datasets=datasets,data_distrib=data_distrib,data_param=data_param,
                                 Nclients=Nclients,seeds=[0,142,242,342,442,542])
        for result in results:
            path = abspath + '\\results' + str(Nclients) + 'clients\\'+ result.index.name
            result.to_csv(path_or_buf=path)

In [ ]:
abspath = os.path.abspath('')
multiclass_datasets = {'nursery':(X_data_nursery,targets_nursery),'car':(X_data_car,y_data_car),
            'covertype':(X_data_covertype,y_data_covertype),'pendig':(X_data_pendig,y_data_pendig)}
numberOfClients = [5,10,20]
for Nclients in numberOfClients:
    results = compare_models_woutXval_myparams(datasets=multiclass_datasets,data_distrib='niid_label_quantity_skew',data_param=3,
                                 Nclients=Nclients,seeds=[0,142,242,342,442,542])
    for result in results:
        path = abspath + '\\results' + str(Nclients) + 'clients\\'+ result.index.name
        result.to_csv(path_or_buf=path)

In [ ]:
bigdatasets = {'bank':(X_data_bank,targets_bank),
            'adult':(X_data_adult,targets_adult),'credit':(X_data_credit,y_data_credit),
            'covertype':(X_data_covertype,y_data_covertype),'magic':(X_data_magic,targets_magic),'pendig':(X_data_pendig,y_data_pendig)}
numberOfClients = [50,100]
data_distribution = [('iid',None),('niid_quantity_skew',0.5),('niid_dirichlet_label_skew',0.5)]
for Nclients in numberOfClients:
    for data_distrib,data_param in data_distribution:
        results = compare_models_woutXval_myparams(datasets=datasets,data_distrib=data_distrib,data_param=data_param,
                                 Nclients=Nclients,seeds=[0,142,242,342,442,542])
        for result in results:
            path = abspath + '\\results' + str(Nclients) + 'clients\\'+ result.index.name
            result.to_csv(path_or_buf=path)

In [ ]:
abspath = os.path.abspath('')
big_multiclass_datasets = {
            'covertype':(X_data_covertype,y_data_covertype),'pendig':(X_data_pendig,y_data_pendig)}
numberOfClients = [50,100]
for Nclients in numberOfClients:
    results = compare_models_woutXval_myparams(datasets=big_multiclass_datasets,data_distrib='niid_label_quantity_skew',data_param=3,
                                 Nclients=Nclients,seeds=[0,142,242,342,442,542])
    for result in results:
        path = abspath + '\\results' + str(Nclients) + 'clients\\'+ result.index.name
        result.to_csv(path_or_buf=path)